# Full ML Pipeline - Real Estate Price Prediction

This notebook runs the **complete** training workflow: Dataset → Data Cleaning → EDA → Feature Engineering → Preprocessing Pipeline → Train Multiple Models → Model Evaluation → Hyperparameter Tuning → Select Best Model → Save Model.

The final model is saved to **`models/house_price_model.pkl`** and is then consumed by the FastAPI backend (`app/main.py`) and the Streamlit frontend (`app/app.py`).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

from src.config import (
    DATA_DIR, RAW_DATA_PATH, MODELS_DIR, MODEL_PATH, METRICS_PATH,
    RANDOM_STATE, TEST_SIZE, CATEGORICAL_FEATURES, NUMERICAL_FEATURES,
)
from src.evaluation import compute_metrics, format_comparison_table
from src.preprocessing import FeatureEngineering, build_full_pipeline

sns.set_theme(style="whitegrid")
print("Setup complete. Project root:", PROJECT_ROOT)

## 2. Dataset

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
print("Shape:", df.shape)
df.head()

## 3. Data Cleaning

In [ ]:
df = df.drop(columns=["Property_ID"])
print("Missing values:\n", df.isna().sum().sum(), "missing")
print("Duplicate records:", int(df.duplicated().sum()))
print("\nData types:\n", df.dtypes)
print("\nSummary statistics:\n", df.describe().T)

## 4. Exploratory Data Analysis

### 4.1 Summary statistics

In [ ]:
print("Numerical columns:")
display(df.select_dtypes(include="number").describe().T)
for col in df.select_dtypes(include=["object", "str"]).columns:
    print(f"\n{col}:")
    display(df[col].value_counts())

### 4.2 Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(df["Price"], kde=True, ax=axes[0, 0]); axes[0, 0].set_title("Price distribution")
sns.scatterplot(data=df, x="Area", y="Price", alpha=0.6, ax=axes[0, 1]); axes[0, 1].set_title("Area vs Price")
sns.boxplot(data=df, x="Bedrooms", y="Price", ax=axes[1, 0]); axes[1, 0].set_title("Bedrooms vs Price")
sns.boxplot(data=df, x="Location", y="Price", ax=axes[1, 1]); axes[1, 1].set_title("Location vs Price")
axes[1, 1].tick_params(axis="x", rotation=15)
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include="number").corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation heatmap"); plt.show()

## 5. Feature Engineering

In [ ]:
eng = FeatureEngineering(enable=True)
sample = eng.fit_transform(df.head())
print("Engineered columns:", [c for c in sample.columns if c not in df.columns])
print("\nAvailable engineered features:")
print(["Area_per_Bedroom", "Total_Rooms", "Age_Group", "Has_Garden_Room_Budget"])
sample.head()

## 6. Preprocessing Pipeline

In [ ]:
X = df.drop(columns=["Price"])
y = df["Price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print("Train size:", X_train.shape, "| Test size:", X_test.shape)

# A pipeline is built with build_full_pipeline(model, use_engineering=True/False)
# so the exact same preprocessing is applied during training and prediction.
print("Example pipeline (Linear Regression + engineering):")
print(build_full_pipeline(LinearRegression()))

## 7. Train Multiple Models

In [ ]:
MODELS = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=RANDOM_STATE, verbosity=0),
}

results = []
for name, model in MODELS.items():
    for use_eng in (True, False):
        pipe = build_full_pipeline(model, use_engineering=use_eng)
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        m = compute_metrics(y_test, y_pred)
        m.update({"model": name, "engineering": "yes" if use_eng else "no"})
        results.append(m)
        print(f"Trained {name:<18} eng={use_eng}  R2={m['R2']:.4f}  RMSE={m['RMSE']:,.0f}")

## 8. Model Evaluation

In [ ]:
print(format_comparison_table(results))

results_df = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(10, 5))
order = results_df.sort_values("RMSE")["model"].tolist()
sns.barplot(data=results_df.sort_values("RMSE"), x="RMSE", y="model", hue="engineering", ax=ax)
ax.set_title("RMSE by model (lower is better)")
plt.tight_layout(); plt.show()

## 9. Hyperparameter Tuning

In [ ]:
best = min(results, key=lambda r: r["RMSE"])
print("Best model:", best["model"], "| engineering:", best["engineering"])

base_model = MODELS[best["model"]]
pipe = build_full_pipeline(base_model, use_engineering=best["engineering"] == "yes")

if best["model"] == "Random Forest":
    param_grid = {
        "model__n_estimators": [100, 300],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_split": [2, 5],
    }
else:
    param_grid = {
        "model__n_estimators": [100, 300],
        "model__learning_rate": [0.05, 0.1, 0.2],
        "model__max_depth": [3, 6],
    }

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV RMSE: {:,}".format(round(-grid.best_score_, 2)))

## 10. Select Best Model

In [ ]:
best_pipeline = grid.best_estimator_
y_pred = best_pipeline.predict(X_test)
final_metrics = compute_metrics(y_test, y_pred)
print("Final test metrics for tuned", best["model"], ":")
for k, v in final_metrics.items():
    print(f"  {k:<6}: {v:,.4f}")

## 11. Save Model

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, MODEL_PATH)
print("Saved pipeline ->", MODEL_PATH)

metadata = {
    "best_model": best["model"],
    "feature_engineering": best["engineering"],
    "test_metrics": final_metrics,
    "train_size": len(X_train),
    "test_size": len(X_test),
}
METRICS_PATH.write_text(__import__("json").dumps(metadata, indent=2))
print("Saved metrics ->", METRICS_PATH)

# Verify: load the saved pipeline and predict on a fresh raw property
loaded = joblib.load(MODEL_PATH)
demo = pd.DataFrame([{
    "Area": 3000, "Bedrooms": 3, "Bathrooms": 2, "Age": 10,
    "Location": "City Center", "Property_Type": "Villa",
}])
print("Example prediction:", f"{loaded.predict(demo)[0]:,.0f} PKR")